In [1]:
%pip install gliner bert_score sentence_transformers osmnx


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download('stopwords')
import spacy

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ishankinikar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
import json
import pickle
import osmnx as ox
import networkx as nx
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from eval.evaluation import prompt_based_route_evaluator
from routing.NER import GlinerAStarRouter
from routing.synset import OSMSemanticBridge

# The Schema defines the technical 'Search Space' for the Bridge
tag_schema = {
    "continuous": {
        "maxspeed_imputed": {"min": 10, "max": 65, "unit": "mph"},
        "lanes": {"min": 1, "max": 6},
        "length": {"min": 2, "max": 6845}
    },
    "discrete": {
        "highway": ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 
                     'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
        "access": ["yes", "no"],
        "bridge": ["yes", "viaduct"],
        "junction": ["roundabout", "jughandle"],
        "ref": ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'I 91', 'I 90', 'MA 116']
    }
}

if __name__ == "__main__":
    mode = 'astar'
    
    # 1. Load Data
    graph_path = "research/pioneer_valley_v2.pkl"
    with open(graph_path, "rb") as f:
        G = pickle.load(f)

    with open("research/synthetic_dataset.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

    # 2. Initialize Models
    # Actor/Bridge model (Efficiency focused)
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # NEW: Initialize the Bridge with the Schema instead of raw Graph Crawling
    # This prevents the 'accidental' mapping of noisy graph strings
    bridge = OSMSemanticBridge(tag_schema, st_model, threshold=0.80)
    
    # Initialize the Router with high-contrast weights for better Dijkstra/A* differentiation
    # [Penalty Multiplier, Reward Multiplier]
    router = GlinerAStarRouter(G, [25.0, 0.01], bridge)
    
    gen_routes, gen_prompts = [], []

    print(f"\n--- Running Neurosymbolic Experiment (Mode: {mode}) ---")
    # Limiting batch for testing: 100:210
    for item in tqdm(data[100:200]):
        try:
            # Geocode to lat/lon with specific regional context
            s_pt = ox.geocoder.geocode(f"{item['start']}, MA, USA")
            e_pt = ox.geocoder.geocode(f"{item['end']}, MA, USA")
            
            s_node = ox.distance.nearest_nodes(G, X=s_pt[1], Y=s_pt[0])
            e_node = ox.distance.nearest_nodes(G, X=e_pt[1], Y=e_pt[0])
            
            if s_node == e_node: continue
            if not nx.has_path(G, s_node, e_node): continue

            # Routing execution using our GLiNER + Schema logic
            route = router.find_route(s_node, e_node, item['prompt'], algorithm=mode)
            
            if route:
                gen_routes.append(route)
                gen_prompts.append(item['prompt'])
                
        except Exception:
            continue

    # 3. Final Evaluation
    if gen_routes:
        print(f"\nSuccessfully generated {len(gen_routes)} routes. Evaluating...")
        # The Evaluator now uses spaCy and NLTK internally for the 'Better Way' logic
        evaluator = prompt_based_route_evaluator(G, gen_prompts, gen_routes, bridge)
        results = evaluator.evaluate_method()
        
        print("\n--- Final Consolidated Metrics ---")
        for metric, value in results.items():
            # Formatting the print to handle the new satisfaction scores
            print(f"{metric:30}: {value:.4f}")
            
        # Optional: Print a single prompt/route logic check
        print(f"\nSample Prompt: {gen_prompts[0]}")
    else:
        print("\n[!] No valid routes were generated. Check graph connectivity or geocoding.")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 643.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bridge: Building Schema-Grounded Index for categories: ['highway', 'access', 'bridge', 'junction', 'ref']


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 57456.22it/s]



--- Running Neurosymbolic Experiment (Mode: astar) ---


100%|██████████| 100/100 [02:33<00:00,  1.53s/it]



Successfully generated 100 routes. Evaluating...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1246.39it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1059.88it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bi


--- Final Consolidated Metrics ---
avg_path_validity             : 1.0000
avg_deviation_penalty         : 1.0028
avg_constraint_satisfaction   : 0.4245
avg_semantic_alignment_f1     : 0.7914

Sample Prompt: Take me from Holyoke to Greenfield on a local road.
